In [1]:
!pip -q install langchain-groq langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00


In [2]:
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=getpass("Enter your Groq API key: ")
)

print("Groq model initialized successfully!")

Enter your Groq API key: ··········
Groq model initialized successfully!


**Exercise 1: Customer Review to Support Ticket Pipeline**

Define the Pydantic schema

In [3]:
from pydantic import BaseModel, Field

class ReviewData(BaseModel):
    complaint: str = Field(description="Core customer complaint")
    product_or_feature: str = Field(description="Product or feature mentioned")
    sentiment: str = Field(description="Customer sentiment")

Step 2 — Create the extraction pipeline

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

review_parser = PydanticOutputParser(pydantic_object=ReviewData)

review_prompt = PromptTemplate(
    template="""
Extract the core complaint, product or feature, and customer sentiment
from the customer review below.

Customer review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": review_parser.get_format_instructions()
    }
)

review_chain = review_prompt | llm | review_parser

Step 3 — Create the support ticket writer

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ticket_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a support ticket writer. Create a concise internal "
        "support ticket summary using only the structured review data provided."
    ),
    (
        "human",
        """
Create a support ticket containing:
- Complaint
- Product or feature
- Customer sentiment
- Suggested next action for the support team

Structured review data:
{structured_data}
"""
    )
])

ticket_chain = ticket_prompt | llm | StrOutputParser()

Step 4 — Connect both steps and run

In [6]:
review_text = """
I bought the SoundBeat wireless headphones last week.
The sound quality is good, but the left earbud stops working
after about ten minutes. I am disappointed with this issue.
"""

# Step 1: Extract structured data
review_data = review_chain.invoke({"review_text": review_text})

print("STEP 1 — Structured Review Data")
print(review_data.model_dump())

# Step 2: Use only Step 1's structured output
ticket_summary = ticket_chain.invoke({
    "structured_data": review_data.model_dump_json()
})

print("\nSTEP 2 — Support Ticket Summary")
print(ticket_summary)

STEP 1 — Structured Review Data
{'complaint': 'Left earbud stops working after about ten minutes', 'product_or_feature': 'SoundBeat wireless headphones (left earbud)', 'sentiment': 'disappointed'}

STEP 2 — Support Ticket Summary
**Support Ticket Summary**

- **Complaint:** Left earbud stops working after about ten minutes.  
- **Product/Feature:** SoundBeat wireless headphones (left earbud).  
- **Customer Sentiment:** Disappointed.  
- **Suggested Next Action:** Initiate a diagnostic and offer a replacement or repair for the left earbud; request the customer to provide a short video or test log if possible.


**Exercise 2: Research Paper Abstract to Layperson Summary Pipeline**

Step 1 — Define the Pydantic schema

In [7]:
from pydantic import BaseModel, Field

class ResearchData(BaseModel):
    research_question: str = Field(
        description="Main question the research investigates"
    )
    method: str = Field(
        description="Method used in the research"
    )
    key_finding: str = Field(
        description="Main finding reported by the research"
    )

Step 2 — Create the extraction pipeline

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

research_parser = PydanticOutputParser(
    pydantic_object=ResearchData
)

research_prompt = PromptTemplate(
    template="""
Extract the research question, method, and key finding
from the abstract below. Do not add information that is not stated.

Research abstract:
{abstract_text}

{format_instructions}
""",
    input_variables=["abstract_text"],
    partial_variables={
        "format_instructions":
            research_parser.get_format_instructions()
    }
)

research_chain = research_prompt | llm | research_parser

Step 3 — Create the layperson summary pipeline

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a science communicator explaining research "
        "to a general audience. Use simple, clear language "
        "and avoid unnecessary scientific jargon."
    ),
    (
        "human",
        """
Explain the research using only the structured information below.

Research question: {research_question}
Method: {method}
Key finding: {key_finding}

Write one short paragraph that a non-expert can understand.
Do not invent details or conclusions.
"""
    )
])

summary_chain = summary_prompt | llm | StrOutputParser()

Step 4 — Run the complete pipeline

In [10]:
abstract_text = """
This study investigated whether regular physical activity
improves sleep quality among university students. Researchers
surveyed 200 students about their weekly exercise habits and
sleep quality. The results showed that students who reported
regular physical activity also tended to report better sleep
quality. The study was observational and does not establish
that exercise directly caused the improvement.
"""

# Step 1: Extract structured research information
research_data = research_chain.invoke({
    "abstract_text": abstract_text
})

print("STEP 1 — Structured Research Data")
print(research_data.model_dump())

# Step 2: Summarize using only extracted information
layperson_summary = summary_chain.invoke({
    "research_question": research_data.research_question,
    "method": research_data.method,
    "key_finding": research_data.key_finding
})

print("\nSTEP 2 — Layperson Summary")
print(layperson_summary)

STEP 1 — Structured Research Data
{'research_question': 'Whether regular physical activity improves sleep quality among university students', 'method': 'Surveyed 200 students about their weekly exercise habits and sleep quality', 'key_finding': 'Students who reported regular physical activity also tended to report better sleep quality'}

STEP 2 — Layperson Summary
Researchers wanted to know if doing regular exercise helps university students sleep better. They asked 200 students about how much they exercise each week and how well they sleep. The results showed that students who reported exercising regularly also tended to say they slept better.


**Exercise 3: Interview Transcript to Candidate Scorecard Pipeline**

Step 1 — Define the Pydantic schemas

In [11]:
from pydantic import BaseModel, Field


class InterviewAnswer(BaseModel):
    question: str = Field(description="Interview question")
    answer: str = Field(description="Candidate's answer")


class ExtractedInterview(BaseModel):
    answers: list[InterviewAnswer]


class SkillEvaluation(BaseModel):
    skill: str = Field(description="Skill being assessed")
    demonstrated: bool = Field(
        description="True only when the answer provides clear evidence"
    )
    evidence_or_gap: str = Field(
        description="Evidence from the answer or explanation of missing evidence"
    )


class InterviewEvaluation(BaseModel):
    evaluations: list[SkillEvaluation]


class ScorecardRow(BaseModel):
    skill: str
    demonstrated: bool
    evidence_or_gap: str


class CandidateScorecard(BaseModel):
    scorecard_rows: list[ScorecardRow]
    overall_recommendation: str = Field(
        description="Evidence-based hiring recommendation"
    )

Step 2 — Extract the interview answers

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

interview_parser = PydanticOutputParser(
    pydantic_object=ExtractedInterview
)

interview_prompt = PromptTemplate(
    template="""
Extract every interview question and its corresponding candidate answer
from the transcript.

Do not invent answers. If an answer is missing, record that clearly.

Transcript:
{transcript_text}

{format_instructions}
""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions": interview_parser.get_format_instructions()
    }
)

interview_chain = interview_prompt | llm | interview_parser

Step 3 — Evaluate the required skills

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

evaluation_parser = PydanticOutputParser(
    pydantic_object=InterviewEvaluation
)

evaluation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a technical interviewer evaluating skill evidence. "
        "Assess only what the candidate's answers demonstrate. "
        "Do not assume a skill is demonstrated without clear evidence."
    ),
    (
        "human",
        """
Required skills:
{skills_to_assess}

Extracted interview answers:
{extracted_answers}

Evaluate every required skill. For each skill, provide:
- skill
- demonstrated (true or false)
- evidence_or_gap

{format_instructions}
"""
    )
])

evaluation_chain = (
    evaluation_prompt
    | llm
    | evaluation_parser
)

Step 4 — Generate the final scorecard

In [14]:
scorecard_parser = PydanticOutputParser(
    pydantic_object=CandidateScorecard
)

scorecard_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a hiring panel coordinator. Prepare a fair, "
        "evidence-based candidate scorecard. Do not invent evidence."
    ),
    (
        "human",
        """
Candidate skill evaluation:
{evaluation_data}

Create a structured scorecard containing a row for each skill,
its demonstrated status, and evidence or gaps.

Give an overall recommendation based only on the available evidence.
If the evidence is insufficient, say that further assessment is needed.

{format_instructions}
"""
    )
])

scorecard_chain = scorecard_prompt | llm | scorecard_parser

Step 5 — Run the complete pipeline

In [15]:
transcript_text = """
Interviewer: Explain how you would find duplicate values in a list.
Candidate: I would use a set to track values already seen and identify
duplicates when a value appears again.

Interviewer: What is the difference between a list and a tuple in Python?
Candidate: Lists are mutable, while tuples are immutable.

Interviewer: Explain how you would handle an API failure.
Candidate: I would check the response status code and use exception
handling, logging, and appropriate retry logic where suitable.
"""

skills_to_assess = [
    "Problem Solving",
    "Python Fundamentals",
    "API Error Handling",
    "Database Design"
]

# Step 1: Extract questions and answers
extracted_data = interview_chain.invoke({
    "transcript_text": transcript_text
})

print("STEP 1 — Extracted Answers")
print(extracted_data.model_dump())

# Step 2: Evaluate skills using extracted answers
evaluation_data = evaluation_chain.invoke({
    "skills_to_assess": ", ".join(skills_to_assess),
    "extracted_answers": extracted_data.model_dump_json(),
    "format_instructions": evaluation_parser.get_format_instructions()
})

print("\nSTEP 2 — Skill Evaluation")
print(evaluation_data.model_dump())

# Step 3: Generate final scorecard
final_scorecard = scorecard_chain.invoke({
    "evaluation_data": evaluation_data.model_dump_json(),
    "format_instructions": scorecard_parser.get_format_instructions()
})

print("\nSTEP 3 — Final Candidate Scorecard")
print(final_scorecard.model_dump())

STEP 1 — Extracted Answers
{'answers': [{'question': 'Explain how you would find duplicate values in a list.', 'answer': 'I would use a set to track values already seen and identify duplicates when a value appears again.'}, {'question': 'What is the difference between a list and a tuple in Python?', 'answer': 'Lists are mutable, while tuples are immutable.'}, {'question': 'Explain how you would handle an API failure.', 'answer': 'I would check the response status code and use exception handling, logging, and appropriate retry logic where suitable.'}]}

STEP 2 — Skill Evaluation
{'evaluations': [{'skill': 'Problem Solving', 'demonstrated': True, 'evidence_or_gap': 'Candidate explains using a set to track seen values and identify duplicates, showing a systematic approach to solving the problem.'}, {'skill': 'Python Fundamentals', 'demonstrated': True, 'evidence_or_gap': 'Answers correctly describe list mutability vs. tuple immutability and use a set for duplicate detection, demonstrating

**Exercise 4: Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline**

Step 1 — Define the Pydantic schemas

In [16]:
from pydantic import BaseModel, Field

class EventDetails(BaseModel):
    event_type: str
    event_date: str
    location: str
    indoor_outdoor_preference: str

class WeatherData(BaseModel):
    temperature: float
    precipitation_chance: float
    conditions: str

Step 2 — Extract event details

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

event_parser = PydanticOutputParser(pydantic_object=EventDetails)

event_prompt = PromptTemplate(
    template="""
Extract the event type, planned date, location, and indoor/outdoor
preference from the description.

Use the date as YYYY-MM-DD when it is explicitly available.
Do not invent missing details. If a detail is missing, use "Not specified".

Event description:
{event_description_text}

{format_instructions}
""",
    input_variables=["event_description_text"],
    partial_variables={
        "format_instructions": event_parser.get_format_instructions()
    }
)

event_chain = event_prompt | llm | event_parser

Configure the weather API

In [18]:
%pip install -q requests

In [19]:
import requests
from datetime import date, timedelta
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

weather_parser = PydanticOutputParser(pydantic_object=WeatherData)

weather_query_prompt = PromptTemplate(
    template="""
Prepare a weather API query using these extracted event details.

Location: {location}
Date: {event_date}

Return the location and date exactly as supplied.
Do not invent coordinates or weather values.
""",
    input_variables=["location", "event_date"]
)

def get_weather(location, event_date):
    # Geocode the extracted location
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=20
    )
    geo_response.raise_for_status()
    places = geo_response.json().get("results", [])

    if not places:
        raise ValueError(f"Location not found: {location}")

    place = places[0]
    latitude = place["latitude"]
    longitude = place["longitude"]

    # Fetch forecast for the selected date
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": "temperature_2m_max,precipitation_probability_max,weather_code",
            "timezone": "auto",
            "start_date": event_date,
            "end_date": event_date
        },
        timeout=20
    )
    weather_response.raise_for_status()
    data = weather_response.json().get("daily", {})

    if not data.get("time"):
        raise ValueError(
            "No forecast available for this date. "
            "Choose a date within the API's forecast range."
        )

    code = data["weather_code"][0]
    descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Slight snow",
        73: "Moderate snow",
        75: "Heavy snow",
        80: "Rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        95: "Thunderstorm"
    }

    raw_weather = {
        "temperature": data["temperature_2m_max"][0],
        "precipitation_chance": data["precipitation_probability_max"][0],
        "conditions": descriptions.get(code, f"Weather code {code}")
    }

    # Validate/normalize the API response using Pydantic
    return weather_parser.pydantic_object(**raw_weather)

Step 4 — Generate the event recommendation

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

recommendation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an event planning assistant. Give practical, cautious "
        "weather-based advice to event organizers. Do not claim certainty "
        "beyond the supplied forecast."
    ),
    (
        "human",
        """
Event details:
{event_details}

Weather data:
{weather_data}

Recommend one of these actions:
- Proceed as planned
- Reschedule
- Move indoors

Explain the recommendation briefly using the supplied details.
If the forecast is uncertain or unsuitable, mention that the organizer
should check an updated forecast before making a final decision.

Return only the recommendation message.
"""
    )
])

recommendation_chain = recommendation_prompt | llm | StrOutputParser()

Step 5 — Run the complete pipeline

In [21]:
event_date_example = (date.today() + timedelta(days=7)).isoformat()

event_description_text = f"""
We are organizing a college sports day in Pune, Maharashtra,
on {event_date_example}. Most activities will take place outdoors.
We would prefer to hold the event outside, but moving indoors
is possible if the weather is unsuitable.
"""

# Step 1: Extract event details
event_data = event_chain.invoke({
    "event_description_text": event_description_text
})

print("STEP 1 — Event Details")
print(event_data.model_dump())

# Step 2: Format the weather query and fetch real API data
weather_query = weather_query_prompt.invoke({
    "location": event_data.location,
    "event_date": event_data.event_date
})

print("\nWeather query prepared:")
print(weather_query.to_string())

weather_data = get_weather(
    event_data.location,
    event_data.event_date
)

print("\nSTEP 2 — Structured Weather Data")
print(weather_data.model_dump())

# Step 3: Generate recommendation from the two structured objects
recommendation = recommendation_chain.invoke({
    "event_details": event_data.model_dump_json(),
    "weather_data": weather_data.model_dump_json()
})

print("\nSTEP 3 — Event Recommendation")
print(recommendation)

STEP 1 — Event Details
{'event_type': 'college sports day', 'event_date': '2026-10-29', 'location': 'Pune, Maharashtra', 'indoor_outdoor_preference': 'Outdoor (preferred, indoor possible if weather unsuitable)'}

Weather query prepared:

Prepare a weather API query using these extracted event details.

Location: Pune, Maharashtra
Date: 2026-10-29

Return the location and date exactly as supplied.
Do not invent coordinates or weather values.


STEP 2 — Structured Weather Data
{'temperature': 32.3, 'precipitation_chance': 25.0, 'conditions': 'Mainly clear'}

STEP 3 — Event Recommendation
Proceed as planned. The forecast indicates mainly clear skies with only a 25% chance of precipitation and a comfortable temperature of 32.3 °C, which is suitable for outdoor activities. Keep an eye on updates closer to the event date.


**Exercise 5: Customer Support Chat Log to Escalation Report Pipeline**

Step 1: Extract structured information

In [22]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class SupportSummary(BaseModel):
    issue: str = Field(description="Main problem reported by the customer")
    customer_tone: str = Field(description="Customer's emotional tone")
    resolution_attempted: str = Field(description="Troubleshooting steps already attempted")

support_summary_parser = PydanticOutputParser(
    pydantic_object=SupportSummary
)

support_summary_prompt = PromptTemplate(
    template="""
You are a customer support analyst.
Extract the important information from the support chat log.

If a detail is missing, clearly state "Not provided".
Do not invent information.

Support chat log:
{chat_log_text}

{format_instructions}
""",
    input_variables=["chat_log_text"],
    partial_variables={
        "format_instructions": support_summary_parser.get_format_instructions()
    }
)

support_summary_chain = support_summary_prompt | llm | support_summary_parser

Step 2: Check escalation readiness

In [23]:
from langchain_core.prompts import ChatPromptTemplate

class EscalationReadiness(BaseModel):
    missing_fields: list[str] = Field(
        description="Information missing for escalation"
    )
    is_ready_to_escalate: bool = Field(
        description="Whether the issue has enough information for escalation"
    )

readiness_parser = PydanticOutputParser(
    pydantic_object=EscalationReadiness
)

readiness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an escalation-readiness reviewer.
Check whether the support issue contains enough information for escalation.

Check for:
- Customer account ID
- Error code or exact error message
- Affected product or service
- Troubleshooting already attempted
- When the issue started

List missing information in missing_fields.
Set is_ready_to_escalate to true only when sufficient information exists
for the technical team to investigate. Do not invent details.

{format_instructions}"""
    ),
    (
        "human",
        "Review this structured support summary:\n{support_summary}"
    )
])

readiness_prompt = readiness_prompt.partial(
    format_instructions=readiness_parser.get_format_instructions()
)

readiness_chain = readiness_prompt | llm | readiness_parser

Step 3: Generate the escalation report

In [24]:
from langchain_core.output_parsers import StrOutputParser

escalation_report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a support operations lead.
Create a clear, prioritized escalation report for the technical team.

Include:
1. Priority level and reason
2. Customer issue and tone
3. Troubleshooting already attempted
4. Missing information
5. Recommended next actions

If important details are missing, explicitly mark them as pending.
Do not invent customer details."""
    ),
    (
        "human",
        """Structured support summary:
{support_summary}

Escalation readiness review:
{readiness_review}

Prepare the final escalation report."""
    )
])

escalation_report_chain = (
    escalation_report_prompt | llm | StrOutputParser()
)

Step 4: Run the complete pipeline

In [25]:
chat_log_text = """
Customer: My payment is failing in the mobile app.
Support: What error message do you see?
Customer: It just says payment failed. I have tried reinstalling
the app and logging in again, but it still does not work.
Support: Please provide your account ID and the exact error code.
Customer: I don't have that information right now. This is frustrating
because I need to complete my payment today.
"""

# Step 1: Extract structured information
summary = support_summary_chain.invoke({
    "chat_log_text": chat_log_text
})

print("STEP 1: STRUCTURED SUPPORT SUMMARY")
print(summary.model_dump_json(indent=2))

# Step 2: Check escalation readiness
readiness = readiness_chain.invoke({
    "support_summary": summary.model_dump_json(indent=2)
})

print("\nSTEP 2: ESCALATION READINESS")
print(readiness.model_dump_json(indent=2))

# Step 3: Generate the final report
report = escalation_report_chain.invoke({
    "support_summary": summary.model_dump_json(indent=2),
    "readiness_review": readiness.model_dump_json(indent=2)
})

print("\nSTEP 3: FINAL ESCALATION REPORT")
print(report)

STEP 1: STRUCTURED SUPPORT SUMMARY
{
  "issue": "Payment is failing in the mobile app.",
  "customer_tone": "Frustrated",
  "resolution_attempted": "Reinstalling the app and logging in again."
}

STEP 2: ESCALATION READINESS
{
  "missing_fields": [
    "Customer account ID",
    "Error code or exact error message",
    "When the issue started"
  ],
  "is_ready_to_escalate": false
}

STEP 3: FINAL ESCALATION REPORT
**Escalation Report – Technical Team**

| Item | Details |
|------|---------|
| **Priority Level** | **P2 – High** |
| **Reason for Priority** | Payment failure in the mobile app directly impacts revenue and customer trust. The customer is frustrated and has already attempted basic troubleshooting with no resolution. Prompt attention is required to prevent churn. |
| **Customer Issue & Tone** | *Issue*: “Payment is failing in the mobile app.”  <br>*Tone*: Frustrated. |
| **Troubleshooting Already Attempted** | 1. Reinstalled the app.<br>2. Logged in again. |
| **Missing Infor

**Exercise 6: Iterative Content Refinement Pipeline**

Step 1: Generate the initial content

In [26]:
from langchain_core.prompts import PromptTemplate

content_prompt = PromptTemplate(
    template="""
You are a professional technical content writer.

Write a short explanation of the following topic:
{topic}

Target audience: {audience}
Maximum length: {max_words} words

Make the explanation clear, accurate, and easy to understand.
""",
    input_variables=["topic", "audience", "max_words"]
)

content_chain = content_prompt | llm | StrOutputParser()

topic = "Benefits of cloud computing for small businesses"
audience = "College students"
max_words = 150

initial_content = content_chain.invoke({
    "topic": topic,
    "audience": audience,
    "max_words": max_words
})

print("STEP 1: INITIAL CONTENT")
print(initial_content)

STEP 1: INITIAL CONTENT
**Benefits of Cloud Computing for Small Businesses**

Cloud computing lets small businesses access powerful software and storage over the internet instead of buying expensive hardware.  
* **Cost savings** – Pay only for what you use; no large upfront IT investments or maintenance fees.  
* **Scalability** – Quickly add or remove resources (storage, computing power) as your business grows or slows.  
* **Accessibility** – Work from anywhere with an internet connection, enabling remote teams and mobile workers.  
* **Automatic updates** – Software providers handle upgrades, keeping security and features current without extra effort.  
* **Disaster recovery** – Data is stored off‑site, so a local outage or hardware failure doesn’t erase your information.  

Overall, the cloud gives small firms the flexibility, affordability, and resilience that once required big budgets and IT departments.


Step 2: Evaluate the content

In [27]:
class ContentReview(BaseModel):
    meets_requirements: bool = Field(
        description="Whether the content satisfies all requirements"
    )
    issues: list[str] = Field(
        description="Problems or weaknesses that need improvement"
    )
    improvement_suggestions: list[str] = Field(
        description="Specific suggestions for improving the content"
    )

content_review_parser = PydanticOutputParser(
    pydantic_object=ContentReview
)

review_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a strict content quality reviewer.
Evaluate the supplied content against its topic, target audience,
maximum word limit, clarity, and accuracy.

Identify genuine weaknesses. Do not invent issues.
{format_instructions}"""
    ),
    (
        "human",
        """Topic: {topic}
Target audience: {audience}
Maximum words: {max_words}

Content to review:
{content}"""
    )
])

review_prompt = review_prompt.partial(
    format_instructions=content_review_parser.get_format_instructions()
)

review_chain = review_prompt | llm | content_review_parser

review = review_chain.invoke({
    "topic": topic,
    "audience": audience,
    "max_words": max_words,
    "content": initial_content
})

print("STEP 2: CONTENT REVIEW")
print(review.model_dump_json(indent=2))

STEP 2: CONTENT REVIEW
{
  "meets_requirements": true,
  "issues": [
    "The content does not explicitly speak to the college student audience; it could include references to student projects or startups to improve relevance.",
    "Security is only briefly mentioned; a short note on data encryption or compliance would strengthen the message."
  ],
  "improvement_suggestions": [
    "Add a brief example of a student-led business or project that benefits from cloud computing.",
    "Include a short statement about data encryption and compliance to address security concerns.",
    "Use a hook or phrasing that directly addresses college students' interests and challenges."
  ]
}


Step 3: Refine the content

In [28]:
refinement_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert content editor.
Improve the supplied draft using the review feedback.

Preserve correct information, fix identified weaknesses,
follow the topic and audience requirements, and stay within
the specified word limit. Return only the refined content."""
    ),
    (
        "human",
        """Topic: {topic}
Target audience: {audience}
Maximum words: {max_words}

Original content:
{content}

Review feedback:
{review_feedback}

Produce the refined final version."""
    )
])

refinement_chain = refinement_prompt | llm | StrOutputParser()

final_content = refinement_chain.invoke({
    "topic": topic,
    "audience": audience,
    "max_words": max_words,
    "content": initial_content,
    "review_feedback": review.model_dump_json(indent=2)
})

print("STEP 3: REFINED FINAL CONTENT")
print(final_content)

STEP 3: REFINED FINAL CONTENT
**Why College‑Student Startups Love Cloud Computing**

Think of your next group project or campus startup—no need to buy a server room. Cloud computing gives you instant, on‑demand software and storage over the internet, just like the tools you already use for class.

* **Cost‑effective** – Pay only for what you use; no pricey hardware or hidden maintenance fees.  
* **Scalable** – Add or drop storage and computing power in minutes as your project grows or slows.  
* **Anywhere access** – Work from the dorm, library, or a coffee shop—perfect for remote teams.  
* **Auto‑updates & security** – Providers handle patches, and data is encrypted at rest and in transit, meeting major compliance standards.  
* **Built‑in disaster recovery** – Off‑site backups mean a laptop crash or campus outage won’t wipe your work.

For students building a new app or research platform, the cloud delivers the flexibility, affordability, and peace of mind that used to require a fu

Step 4: Compare the initial and refined versions

In [29]:
print("=" * 50)
print("INITIAL CONTENT")
print("=" * 50)
print(initial_content)

print("\n" + "=" * 50)
print("REFINED CONTENT")
print("=" * 50)
print(final_content)

print("\nInitial word count:", len(initial_content.split()))
print("Refined word count:", len(final_content.split()))

INITIAL CONTENT
**Benefits of Cloud Computing for Small Businesses**

Cloud computing lets small businesses access powerful software and storage over the internet instead of buying expensive hardware.  
* **Cost savings** – Pay only for what you use; no large upfront IT investments or maintenance fees.  
* **Scalability** – Quickly add or remove resources (storage, computing power) as your business grows or slows.  
* **Accessibility** – Work from anywhere with an internet connection, enabling remote teams and mobile workers.  
* **Automatic updates** – Software providers handle upgrades, keeping security and features current without extra effort.  
* **Disaster recovery** – Data is stored off‑site, so a local outage or hardware failure doesn’t erase your information.  

Overall, the cloud gives small firms the flexibility, affordability, and resilience that once required big budgets and IT departments.

REFINED CONTENT
**Why College‑Student Startups Love Cloud Computing**

Think of yo